# Simulation ALS Data: TAM3C2 + 4D-OBC Analysis

Time-Adaptive M3C2 using the new `py4dgeo.tam3c2` module on a simulated dataset.

**Dataset:** Sand Dune TLS Scans
- **Location:** `C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als`
- **Format:** XYZ files

**Workflow:**
1. Load epochs (timestamps parsed from filenames)
2. Sample corepoints from the reference epoch
3. Build a `TAM3C2` algorithm object and hand it to `SpatiotemporalAnalysis`
4. `analysis.add_epochs(*others)` triggers per-target time-adaptive M3C2
5. Run a custom 4D-OBC region-growing algorithm
6. Visualize aggregation diagnostics + extracted objects

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo
from py4dgeo import (
    TAM3C2,
    Weighting,
    extract_reference_and_others,
    sample_corepoints,
)
from py4dgeo.segmentation import RegionGrowingSeed, temporal_averaging
from py4dgeo.data_loader import read_pc_epochs_and_assign_timestamps

## 1. Configuration

In [ ]:
data_path = r'C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als_downsampled'
output_path = os.path.join(os.getcwd(), 'simulation2_als_downsampled_tam3c2.zip')

reference_timestamp = datetime(2020, 1, 6, 0, 0, 0)
max_epochs_after_reference = None  # Use all available epochs

# TAM3C2 parameters - adjust for sand dune scale and dynamics
normal_radii = [0.5]         # Smaller radii for finer features
max_window_ratio = [0.1]
required_points = 10
cyl_radius = 0.5
max_distance = 10.0
registration_error = 0.01
sigma_ratio = 1.0
space_time_ratio = 1.0
weighting = Weighting.GAUSSIAN
keep_neighborhoods = False

# 4D-OBC parameters
obc_neighborhood_radius = 1
obc_min_segments = 10
obc_minperiod = 3
obc_height_threshold = 0.05
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
obc_smoothing_window = 3

In [ ]:
reference_file_path = os.path.join(os.getcwd(), 'simulation2_reference.zip')
ref_analysis = py4dgeo.SpatiotemporalAnalysis(reference_file_path, force=False)
corepoints = ref_analysis.corepoints.cloud

## 2. Load epochs and filter time range

In [ ]:
epochs = read_pc_epochs_and_assign_timestamps(folder=data_path, start_time=datetime(2020, 1, 1), time_increment=timedelta(days=1))

In [ ]:
# Limit the time range used in the analysis
if max_epochs_after_reference is not None and epochs:
    sorted_eps = sorted(epochs, key=lambda e: e.timestamp)
    ref_idx = next((i for i, e in enumerate(sorted_eps) if e.timestamp == reference_timestamp), None)
    if ref_idx is None:
        raise ValueError(f"Reference {reference_timestamp} not in data")
    epochs = sorted_eps[:ref_idx + 1 + max_epochs_after_reference]
    print(f"Using {len(epochs)} epochs ({epochs[0].timestamp} -> {epochs[-1].timestamp})")

In [ ]:
reference_epoch, other_epochs = extract_reference_and_others(epochs, reference_timestamp)
print(f"Reference: {reference_epoch.timestamp}  ({len(reference_epoch.cloud):,} pts)")
print(f"Other epochs: {len(other_epochs)}")

## 3. Build TAM3C2 and run the spatiotemporal analysis

In [ ]:
tam = TAM3C2(
    epochs_timeseries=epochs,
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    keep_neighborhoods=keep_neighborhoods,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

analysis = py4dgeo.SpatiotemporalAnalysis(output_path, force=True)
analysis.reference_epoch = reference_epoch
analysis.corepoints = corepoints
analysis.m3c2 = tam

analysis.add_epochs(*other_epochs)
print(f"distances shape: {analysis.distances.shape}")
print(f"uncertainties shape: {analysis.uncertainties.shape}")

In [ ]:
# Temporal smoothing for 4D-OBC
analysis.smoothed_distances = temporal_averaging(
    analysis.distances, smoothing_window=obc_smoothing_window
)
print(f"smoothed shape: {analysis.smoothed_distances.shape}")

## 5. Data quality check

In [ ]:
sd = analysis.smoothed_distances
if sd is not None and sd.shape[1] > 0:
    valid_epochs_per_cp = np.sum(~np.isnan(sd), axis=1)
    max_abs_changes = np.nanmax(np.abs(sd), axis=1)

    has_enough = valid_epochs_per_cp >= obc_min_segments
    significant = max_abs_changes >= obc_height_threshold
    both = has_enough & significant

    print(f"Corepoints with >= {obc_min_segments} valid epochs: {has_enough.sum()} ({has_enough.mean()*100:.1f}%)")
    print(f"Corepoints with max |change| >= {obc_height_threshold} m: {significant.sum()} ({significant.mean()*100:.1f}%)")
    print(f"Meeting both: {both.sum()} ({both.mean()*100:.1f}%)")
    print(f"Max change: {np.nanmax(max_abs_changes):.4f} m   Mean change: {np.nanmean(max_abs_changes):.4f} m")
else:
    print("No distance data to analyze.")

## 7. Extract 4D-OBCs

In [ ]:
algo = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=obc_neighborhood_radius,
    min_segments=obc_min_segments,
    minperiod=obc_minperiod,
    height_threshold=obc_height_threshold,
    thresholds=obc_thresholds,
    seed_subsampling=1,
)

analysis.invalidate_results(seeds=True, objects=True, smoothed_distances=False)
objects = algo.run(analysis)
print(f"Extracted {len(objects)} 4D-OBCs from {len(analysis.seeds)} seeds")

## 8. Direct M3C2 Time Series and 4D-OBCs

This baseline uses the same reference epoch, target epochs, corepoints, normal radii, cylinder radius, maximum distance, registration error, temporal smoothing, and 4D-OBC extraction parameters as the TAM3C2 workflow. The difference is that direct M3C2 uses only each target epoch and the reference epoch, without temporal aggregation.

In [ ]:
m3c2_output_path = os.path.join(os.getcwd(), 'simulation2_als_downsampled_m3c2.zip')

m3c2 = py4dgeo.M3C2(
    normal_radii=normal_radii,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

m3c2_analysis = py4dgeo.SpatiotemporalAnalysis(m3c2_output_path, force=True)
m3c2_analysis.reference_epoch = reference_epoch
m3c2_analysis.corepoints = corepoints
m3c2_analysis.m3c2 = m3c2

m3c2_analysis.add_epochs(*other_epochs)
print(f"M3C2 distances shape: {m3c2_analysis.distances.shape}")
print(f"M3C2 uncertainties shape: {m3c2_analysis.uncertainties.shape}")

m3c2_analysis.smoothed_distances = temporal_averaging(
    m3c2_analysis.distances,
    smoothing_window=obc_smoothing_window,
)
print(f"M3C2 smoothed shape: {m3c2_analysis.smoothed_distances.shape}")

m3c2_algo = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=obc_neighborhood_radius,
    min_segments=obc_min_segments,
    minperiod=obc_minperiod,
    height_threshold=obc_height_threshold,
    thresholds=obc_thresholds,
    seed_subsampling=1,
)

m3c2_analysis.invalidate_results(seeds=True, objects=True, smoothed_distances=False)
m3c2_objects = m3c2_algo.run(m3c2_analysis)
print(f"Extracted {len(m3c2_objects)} M3C2 4D-OBCs from {len(m3c2_analysis.seeds)} seeds")
print(f"Saved M3C2 analysis: {m3c2_output_path}")

## 9. Compare GT, M3C2, and TAM3C2 4D-OBCs

The comparison treats every 4D-OBC as a spatiotemporal set: `(corepoint, epoch)` pairs covered by the object's spatial indices and temporal interval. This gives global precision/recall/F1/IoU against GT, plus an object-level matching table.

In [ ]:
# Configure the three analysis archives to compare.
# The M3C2 archive is produced by the Direct M3C2 Time Series cell above.

gt_path = reference_file_path  # ground truth archive
m3c2_path = m3c2_output_path
tam3c2_path = output_path  # current TAM3C2 analysis archive

match_radius = 0.5      # meters; used only if corepoint grids differ
match_iou_threshold = 0.80

analyses = {
    'GT': py4dgeo.SpatiotemporalAnalysis(gt_path, force=False),
    'M3C2': py4dgeo.SpatiotemporalAnalysis(m3c2_path, force=False),
    'TAM3C2': py4dgeo.SpatiotemporalAnalysis(tam3c2_path, force=False),
}

for name, st_analysis in analyses.items():
    objects_for_method = st_analysis.objects
    if objects_for_method is None:
        raise RuntimeError(
            f"{name} analysis has no stored 4D-OBC objects. "
            "Run RegionGrowingAlgorithm on that analysis first."
        )
    print(
        f"{name}: {len(objects_for_method)} objects, "
        f"{st_analysis.corepoints.cloud.shape[0]:,} corepoints, "
        f"distance shape={st_analysis.distances.shape}, "
        f"path={st_analysis.filename}"
    )

In [ ]:
# Diagnostic summary: candidates, seeds, and extracted objects for each archive.
# This helps distinguish three cases:
# 1. no distance signal / too many NaNs,
# 2. seed detection finds no temporal change segments,
# 3. seeds exist but region growing cannot form objects with min_segments.

for name, st_analysis in analyses.items():
    distance_series = st_analysis.smoothed_distances
    if distance_series is None:
        distance_series = st_analysis.distances
    distance_series = np.asarray(distance_series)

    valid_epochs_per_corepoint = np.sum(np.isfinite(distance_series), axis=1)
    with np.errstate(all='ignore'):
        max_abs_change = np.nanmax(np.abs(distance_series), axis=1)

    candidate_mask = (
        (valid_epochs_per_corepoint >= obc_min_segments)
        & (max_abs_change >= obc_height_threshold)
    )
    seeds_for_method = st_analysis.seeds
    objects_for_method = st_analysis.objects

    print(f"\n{name}")
    print(f"  path: {st_analysis.filename}")
    print(f"  distance shape: {distance_series.shape}")
    print(f"  finite ratio: {np.isfinite(distance_series).mean():.3f}")
    print(f"  corepoints with >= {obc_min_segments} valid epochs: {(valid_epochs_per_corepoint >= obc_min_segments).sum():,}")
    print(f"  corepoints with max |change| >= {obc_height_threshold} m: {(max_abs_change >= obc_height_threshold).sum():,}")
    print(f"  corepoints satisfying both simple checks: {candidate_mask.sum():,}")
    print(f"  seeds: {None if seeds_for_method is None else len(seeds_for_method):,}")
    print(f"  objects: {None if objects_for_method is None else len(objects_for_method):,}")

In [ ]:
from scipy.spatial import cKDTree

try:
    import pandas as pd
except ImportError:
    pd = None


def _corepoint_index_mapper(source_analysis, target_analysis, radius):
    source_cp = np.asarray(source_analysis.corepoints.cloud)
    target_cp = np.asarray(target_analysis.corepoints.cloud)

    if len(source_cp) == len(target_cp) and np.allclose(source_cp, target_cp):
        return np.arange(len(source_cp)), np.ones(len(source_cp), dtype=bool)

    tree = cKDTree(target_cp[:, :2])
    dist_xy, mapped = tree.query(source_cp[:, :2], k=1, distance_upper_bound=radius)
    valid = np.isfinite(dist_xy) & (mapped < len(target_cp))
    mapped_safe = np.full(len(source_cp), -1, dtype=int)
    mapped_safe[valid] = mapped[valid]
    print(
        f"Mapped {valid.sum():,}/{len(source_cp):,} source corepoints to GT "
        f"within {radius} m; median distance={np.median(dist_xy[valid]):.3f} m"
    )
    return mapped_safe, valid


def _object_corepoint_set(obj, index_map=None, valid_source=None):
    idx = np.asarray(obj.indices, dtype=int)
    if index_map is None:
        return set(idx.tolist())

    keep = valid_source[idx]
    return set(index_map[idx[keep]].tolist())


def _object_epoch_set(obj):
    return set(range(int(obj.start_epoch), int(obj.end_epoch) + 1))


def _object_token_set(obj, index_map=None, valid_source=None):
    cp_set = _object_corepoint_set(obj, index_map=index_map, valid_source=valid_source)
    epoch_set = _object_epoch_set(obj)
    return {(cp_idx, epoch_idx) for cp_idx in cp_set for epoch_idx in epoch_set}


def _analysis_token_set(objects_for_method, index_map=None, valid_source=None):
    tokens = set()
    for obj in objects_for_method:
        tokens |= _object_token_set(obj, index_map=index_map, valid_source=valid_source)
    return tokens


def _set_metrics(pred_tokens, gt_tokens):
    tp = len(pred_tokens & gt_tokens)
    fp = len(pred_tokens - gt_tokens)
    fn = len(gt_tokens - pred_tokens)
    union = len(pred_tokens | gt_tokens)
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
    iou = tp / union if union else np.nan
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou,
    }


def _iou(a, b):
    union = len(a | b)
    return len(a & b) / union if union else np.nan


def _match_objects(pred_objects, gt_objects, index_map=None, valid_source=None):
    gt_cp_sets = [_object_corepoint_set(obj) for obj in gt_objects]
    gt_epoch_sets = [_object_epoch_set(obj) for obj in gt_objects]
    gt_token_sets = [_object_token_set(obj) for obj in gt_objects]

    rows = []
    for pred_idx, pred_obj in enumerate(pred_objects):
        pred_cp = _object_corepoint_set(pred_obj, index_map=index_map, valid_source=valid_source)
        pred_epochs = _object_epoch_set(pred_obj)
        pred_tokens = _object_token_set(pred_obj, index_map=index_map, valid_source=valid_source)

        best = None
        for gt_idx, (gt_cp, gt_epochs, gt_tokens) in enumerate(zip(gt_cp_sets, gt_epoch_sets, gt_token_sets)):
            st_iou = _iou(pred_tokens, gt_tokens)
            row = {
                'pred_object': pred_idx,
                'gt_object': gt_idx,
                'spatial_iou': _iou(pred_cp, gt_cp),
                'temporal_iou': _iou(pred_epochs, gt_epochs),
                'spacetime_iou': st_iou,
                'pred_corepoints': len(pred_cp),
                'gt_corepoints': len(gt_cp),
                'pred_epochs': len(pred_epochs),
                'gt_epochs': len(gt_epochs),
            }
            if best is None or st_iou > best['spacetime_iou']:
                best = row

        if best is not None:
            best['matched'] = best['spacetime_iou'] >= match_iou_threshold
            rows.append(best)

    return rows


gt_analysis = analyses['GT']
gt_objects = list(gt_analysis.objects)
gt_tokens = _analysis_token_set(gt_objects)

summary_rows = []
match_rows = []
for method in ['M3C2', 'TAM3C2']:
    method_analysis = analyses[method]
    method_objects = list(method_analysis.objects)
    index_map, valid_source = _corepoint_index_mapper(method_analysis, gt_analysis, match_radius)

    method_tokens = _analysis_token_set(
        method_objects,
        index_map=index_map,
        valid_source=valid_source,
    )
    metrics = _set_metrics(method_tokens, gt_tokens)
    metrics.update({
        'method': method,
        'n_objects': len(method_objects),
        'n_gt_objects': len(gt_objects),
        'matched_objects': 0,
    })

    rows = _match_objects(
        method_objects,
        gt_objects,
        index_map=index_map,
        valid_source=valid_source,
    )
    for row in rows:
        row['method'] = method
    metrics['matched_objects'] = sum(row['matched'] for row in rows)

    summary_rows.append(metrics)
    match_rows.extend(rows)

if pd is not None:
    summary_df = pd.DataFrame(summary_rows).set_index('method')
    matches_df = pd.DataFrame(match_rows)
    display(summary_df[['n_objects', 'n_gt_objects', 'matched_objects', 'precision', 'recall', 'f1', 'iou', 'tp', 'fp', 'fn']])
    if len(matches_df) > 0:
        display(matches_df.sort_values(['method', 'spacetime_iou'], ascending=[True, False]))
    else:
        print('No object-level matches to display because no predicted objects or no GT objects were available.')
else:
    summary_df = summary_rows
    matches_df = match_rows
    print('Summary:')
    for row in summary_rows:
        print(row)
    print('Top matches:')
    for row in sorted(match_rows, key=lambda r: r['spacetime_iou'], reverse=True)[:20]:
        print(row)

In [ ]:
# Visual summary of the 4D-OBC comparison.

if pd is not None:
    metrics_to_plot = ['precision', 'recall', 'f1', 'iou']
    ax = summary_df[metrics_to_plot].plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
    ax.set_ylabel('Score')
    ax.set_title('Global 4D-OBC agreement against GT')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    if len(matches_df) > 0 and 'method' in matches_df.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for method, group in matches_df.groupby('method'):
            vals = np.sort(group['spacetime_iou'].to_numpy())[::-1]
            ax.plot(np.arange(1, len(vals) + 1), vals, marker='o', ms=3, lw=1, label=method)
        ax.axhline(match_iou_threshold, color='black', ls='--', lw=1, label=f'match threshold={match_iou_threshold}')
        ax.set_xlabel('Predicted object rank by best GT IoU')
        ax.set_ylabel('Best GT spacetime IoU')
        ax.set_title('Object-level best-match quality')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No object-level matches to plot.')
else:
    print('Install pandas to enable table-based plotting in this cell.')